In [5]:
!pip install -q beir sentence-transformers transformers faiss-cpu gradio

In [6]:
import numpy as np
import pandas as pd
import torch
import faiss

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer

In [7]:
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

data_path = util.download_and_unzip(url, "data")

corpus, queries, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="train")

print("Documents:", len(corpus))
print("Claims:", len(queries))
print("Relevance judgments:", len(qrels))

data/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

  0%|          | 0/5183 [00:00<?, ?it/s]

Documents: 5183
Claims: 809
Relevance judgments: 809


In [1]:
!pip install -q transformers

In [2]:
import torch
from transformers import AutoTokenizer, AutoModel

transformer_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(transformer_name)
transformer = AutoModel.from_pretrained(transformer_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
transformer = transformer.to(device)
transformer.eval()

print("Device:", device)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Device: cuda


In [8]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device: cuda


In [9]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

data_path = util.download_and_unzip(url, "data")

corpus, queries, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="train")

print("Documents:", len(corpus))
print("Claims:", len(queries))

  0%|          | 0/5183 [00:00<?, ?it/s]

Documents: 5183
Claims: 809


In [10]:
claim = queries["0"]

inputs = tokenizer(
    claim,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=256
)

inputs = {key: value.to(device) for key, value in inputs.items()}

print(inputs.keys())
print("Input IDs:", inputs["input_ids"].shape)
print("Attention mask:", inputs["attention_mask"].shape)

print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])
Input IDs: torch.Size([1, 15])
Attention mask: torch.Size([1, 15])
['[CLS]', '0', '-', 'dimensional', 'bio', '##mate', '##rial', '##s', 'lack', 'ind', '##uc', '##tive', 'properties', '.', '[SEP]']


In [11]:
with torch.no_grad():
    outputs = transformer(**inputs)

token_embeddings = outputs.last_hidden_state

print("Token embeddings shape:", token_embeddings.shape)

Token embeddings shape: torch.Size([1, 15, 384])


In [12]:
attention_mask = inputs["attention_mask"]

mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

sum_embeddings = torch.sum(token_embeddings * mask, dim=1)
sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)

sentence_embedding = sum_embeddings / sum_mask

print("Sentence embedding shape:", sentence_embedding.shape)

Sentence embedding shape: torch.Size([1, 384])
